### Plan

1. Download a bunch of World Bank document metadata using API
2. Build ontology using the metadata
3. Build a knowledge graph (KG) by populating ontology with instances of World Bank documents (each row in the metadata)
4. Bring in some additional data from Wikidata
5. Query KG using SPARQL
6. Compare the different ways of interacting with data: using SPARQL queries on the RDF, putting raw metadata into LlamaIndex, and putting the RDF data into LlamaIndex

See accompanying blog post on Medium for full documentation

## 1. Download some World Bank document metadata

In [1]:
import requests
import json
import pandas as pd
import ast

url = 'https://search.worldbank.org/api/v2/wds'
params = {
    'format': 'json',
    'display_title': '"sustainable development"',
    'rows': 100, #Can adjust this to get more/less data
    'page': 1
}
metadata_list = []
for i in range(1):
    response = requests.get(url, params=params)
    data = json.loads(response.content)
    for doc_id in data['documents']:
        metadata = data['documents'][doc_id]
        metadata_list.append(metadata)        
    params['page'] += 1    
df = pd.DataFrame(metadata_list)

In [2]:
#Save raw data to a csv so we can compare results later
df.to_csv("raw.csv")

## 2. Use metadata to create ontology

In [2]:
from rdflib import Graph, RDF, RDFS, Namespace, URIRef, Literal
from rdflib.namespace import SKOS, XSD
from tqdm import tqdm
# Create a new RDF graph
g = Graph()
schema = Namespace('http://schema.org/')
wd = Namespace('http://www.wikidata.org/entity/')
# Define namespaces
prefixes = {
    'schema': schema,
    'wd': wd,
    'skos': SKOS,
    'xsd': XSD
}
for p, ns in prefixes.items():
    g.bind(p, ns)    

In [3]:
import time
import re
import ast
from urllib.parse import quote
from SPARQLWrapper import SPARQLWrapper, JSON
from urllib.error import HTTPError

WDQS_USER_AGENT = "SDKG-notebook/1.0 (https://github.com/; contact: xut565883@gmail.com)"
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}

def make_sparql(endpoint="https://query.wikidata.org/sparql"):
    sparql = SPARQLWrapper(endpoint, agent=WDQS_USER_AGENT)
    return sparql

def escape_sparql_string(s):
    """Escape a value for safe embedding inside a double-quoted SPARQL string literal."""
    return str(s).replace('\\', '\\\\').replace('"', '\\"').replace('\n', ' ').replace('\r', ' ')

def sanitize_uri_component(s):
    """Turn an arbitrary label (which may contain quotes, parens, slashes, etc.) into a valid URI path segment."""
    return quote(str(s).replace(' ', '_'), safe='_-.~')

def safe_literal_eval(s):
    """Like ast.literal_eval, but tolerates bare nan/None tokens left by str()-ing a dict with missing fields."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        fixed = re.sub(r'(?<![\'"\w])nan(?![\'"\w])', 'None', s)
        return ast.literal_eval(fixed)

def run_sparql(sparql, max_retries=6, default_wait=60):
    """Run sparql.query().convert() with retry/backoff on 429 (rate limit) and transient 5xx errors."""
    for attempt in range(max_retries):
        try:
            return sparql.query().convert()
        except HTTPError as e:
            if e.code in RETRYABLE_HTTP_CODES:
                wait = default_wait if e.code == 429 else min(default_wait, 10 * (attempt + 1))
                retry_after = e.headers.get("Retry-After") if e.headers else None
                if retry_after:
                    try:
                        wait = int(retry_after)
                    except ValueError:
                        pass
                print(f"HTTP {e.code}, waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Exceeded max retries due to repeated errors from WDQS")

In [4]:
def get_entity_label(entity_code):
    # Set up the SPARQL endpoint
    sparql = make_sparql()
    # Construct the SPARQL query
    query = f"""
    SELECT ?label WHERE {{
      wd:{entity_code} rdfs:label ?label.
      FILTER (lang(?label) = 'en')
    }}
    """
    # Set the query and response format
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    # Execute the query and retrieve the results
    results = run_sparql(sparql)
    # Extract and return the entity label
    if 'results' in results and 'bindings' in results['results'] and results['results']['bindings']:
        label = results['results']['bindings'][0]['label']['value']
        return label
    return None

In [5]:
def create_subclass_country(column):
    #This puts all entities in this column under the class 'country' when some of them are regions like MENA or continents like Africa. 
    newClass = URIRef(schema + "country")
    g.add((newClass, RDFS.label, Literal("country", lang='en')))
    df[column] = df[column].astype(str)
    for value in df[column].unique():
        if value != "nan":
            # Check Wikidata for a matching class
            sparql = make_sparql()
            query = f"""
                SELECT ?class ?label WHERE {{
                    ?class wdt:P31 wd:Q6256 .
                    ?class rdfs:label "{escape_sparql_string(value)}"@en .
                    OPTIONAL {{ ?class skos:prefLabel ?label FILTER(lang(?label) = "en") }}
                    FILTER(REGEX(STR(?class), "^http://www.wikidata.org/entity/Q[0-9]+$"))
                }}
            """
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            results = run_sparql(sparql)
            # If there is a match, use the Wikidata class as a subclass
            if results['results']['bindings']:
                #Get URI from Wikidata
                uri = results['results']['bindings'][0]['class']['value']
                #Get the 'Q ID' which is the unique ID at the end of the URI
                qid = uri.split('/')[-1]
                country_label = value
                #Create a subclass for each country under the country class
                subclass = URIRef(schema + str(country_label).replace(' ', '_'))
                g.add((subclass, RDF.type, RDFS.Class))
                g.add((subclass, RDFS.subClassOf, newClass))
                # Update the "country_URI" column with the URI for the current country
                df.loc[df[column] == value, "country_URI"] = uri
                uri = URIRef(uri)
                # Define the URI for the new Wikidata URI property
                wd_URI_property = URIRef(schema + "wd_URI")
                # Add the property to the RDF graph
                g.add((wd_URI_property, RDF.type, RDF.Property))
                # Add a label to the property
                label = Literal("Wikidata URI", lang="en")
                g.add((wd_URI_property, RDFS.label, label))
                #Add Wikidata URI as a property to each country class
                g.add((subclass, schema.wd_URI, uri))
                #Add label to each Wikidata Q ID code that it is the Q ID for this particular country
                g.add((uri, RDFS.label, Literal(f"{country_label} wikidata code", lang='en')))
                g.add((subclass, RDFS.label, Literal(str(value), lang='en')))
            else:    
                subclass = URIRef(schema + str(value).replace(' ', '_').replace('-','_'))
                g.add((subclass, RDF.type, RDFS.Class))
                g.add((subclass, RDFS.subClassOf, newClass))
                g.add((subclass, RDFS.label, Literal(str(value), lang='en')))

In [6]:
def create_subclass_world_bank_document(column):
    newClass = URIRef(schema + "world_bank_document")
    g.add((newClass, RDFS.label, Literal("A document produced and written by the World Bank.", lang='en')))   
    df[column] = df[column].astype(str)
    for value in df[column].unique():
        if isinstance(value, str):
            # Check Wikidata for a matching class
            sparql = make_sparql()
            query = f"""
                SELECT ?class ?label WHERE {{
                    ?class rdfs:label "{escape_sparql_string(value)}"@en .
                    OPTIONAL {{ ?class skos:prefLabel ?label FILTER(lang(?label) = "en") }}
                    FILTER(REGEX(STR(?class), "^http://www.wikidata.org/entity/Q[0-9]+$"))
                }}
            """            
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            results = run_sparql(sparql)
            if value != "nan":
                # If there is a match, use the Wikidata class as a subclass
                if results['results']['bindings']:
                    wd_class = results['results']['bindings'][0]['class']['value']
                    #subclass = URIRef(wd_class)
                    qid = wd_class.split('/')[-1]
                    #label = get_entity_label(qid)
                    label = value
                    #print(label)
                    subclass = URIRef(schema + str(label).replace(' ', '_'))
                    #label = Literal(results['results']['bindings'][0]['label']['value']) if 'label' in results['results']['bindings'][0] else Literal(value, lang='en')
                    g.add((subclass, RDF.type, RDFS.Class))
                    #g.add((subclass, RDFS.subClassOf, schema[column]))
                    g.add((subclass, RDFS.subClassOf, newClass))
                    wd_uri = URIRef(wd_class)
                    # Define the URI for the new property
                    wd_URI_property = URIRef(schema + "wd_URI")
                    # Add the property to the RDF graph
                    g.add((wd_URI_property, RDF.type, RDF.Property))
                    # Add a label to the property
                    label = Literal("Wikidata URI", lang="en")
                    g.add((wd_URI_property, RDFS.label, label))
                    g.add((subclass, schema.wd_URI, wd_uri))
                    g.add((wd_uri, RDFS.label, Literal("entity wikidata code", lang='en')))
                    #g.add((subclass, SKOS.prefLabel, Literal(value, lang='en')))
                    g.add((subclass, RDFS.label, Literal(str(value), lang='en')))
                    df[column] = df[column].replace(value, str(subclass))
                else:
                    subclass = URIRef(schema + str(value).replace(' ', '_').replace('-','_'))
                    g.add((subclass, RDF.type, RDFS.Class))
                    g.add((subclass, RDFS.subClassOf, newClass))
                    #g.add((subclass, SKOS.prefLabel, Literal(value, lang='en')))
                    g.add((subclass, RDFS.label, Literal(str(value), lang='en')))
                    df[column] = df[column].replace(value, str(subclass))

In [7]:
def create_subclass_multiple_values(column, names = None):
    newClass = URIRef(schema + str(column))
    df[column] = df[column].astype(str)
    for rowValue in df[column].unique():
        values = rowValue.split(",")
        if names is not None:
            for value in values:
                newID = URIRef(schema + str(column) + "/" + str(value))
                g.add((newID, RDF.type, newClass))
                if value in names:
                    name = names[value]
                    g.add((newID, SKOS.prefLabel, Literal(name, lang='en')))

In [8]:
def create_subclass_trustfund(column, ids = None):
    df['trustfund'] = df['trustfund'].astype(str).str.replace('\n', '').str.replace(" ", "_").str.replace("-", "_")
    newClass = URIRef(schema + "trustfund")
    g.add((newClass, RDFS.label, Literal("trustfund")))
    #RDFLoader doesn't allow a comment unless you also add a label on it
    #g.add((newClass, RDFS.comment, Literal("The World Bank Group (WBG) uses trust funds, a financing arrangement set up with contributions from one or more development partner, to complement core funding from the International Bank for Reconstruction and Development (IBRD), the International Development Association (IDA), and the International Finance Corporation (IFC), in support of the World Bank Group’s goals. Trust funds allow the Bank to mobilize and direct concessional resources to strategic development priorities and to mobilize the resources and capabilities of other development actors through partnership programs.")))
    # Define the id property
    id_property = schema.identifier
    g.add((id_property, RDF.type, RDF.Property))
    g.add((id_property, RDFS.label, Literal("Identifier")))
    #g.add((id_property, RDFS.comment, Literal("The unique identifier of the trustfund.")))
    # Associate the id property with the trustfund class
    g.add((id_property, RDFS.domain, newClass))
    df[column] = df[column].astype(str)
    for rowValue in df[column].unique():
        if rowValue != "nan":
            names = str(rowValue).split(",")
            if ids is not None:
                for name in names:
                    newID = URIRef(schema + "trustfund" + "/" + str(name))
                    g.add((newID, RDF.type, newClass))
                    g.add((newID, RDFS.label, Literal(str(name))))
                    if name in ids:
                        id = ids[name]
                        # Create a URIRef for the trustfund_id resource
                        trustfund_id = URIRef(schema + "trustfund" + "/id/" + Literal(id, datatype=XSD.string))
                        g.add((newID, id_property, trustfund_id))
                        #label = Literal("TEST")
                        label = Literal(f"world bank trustfund ID for {name}", lang="en")
                        g.add((trustfund_id, RDFS.label, label))

In [9]:
def create_subclass_project(column):
    newClass = URIRef(schema + "project")
    g.add((newClass, RDFS.label, Literal("worldbank project")))
    df[column] = df[column].astype(str)
    for value in df[column].unique():
        if value != "nan":
            newID = URIRef(schema + "project/" + str(value).replace(" ","_").replace("-","_"))
            g.add((newID, RDF.type, newClass))
            g.add((newID, RDFS.label, Literal(str(value).replace(" ","_").replace("-","_"), lang='en')))

In [10]:
def create_subclass_authors(column):
    newClass = URIRef(schema + str(column))
    g.add((newClass, RDFS.label, Literal("worldbank authors")))
    df[column] = df[column].astype(str)
    for value in df[column].unique():
        if value != "nan":
            # Add author property for each author
            author_dict = safe_literal_eval(str(value))
            if not isinstance(author_dict, dict):
                continue
            for author_dict_entries in author_dict.values():
                if not isinstance(author_dict_entries, dict):
                    continue
                author_name = author_dict_entries.get('author')
                if not isinstance(author_name, str):
                    continue
                author_uri = URIRef(schema + "author/" + author_name.replace(" ", "_"))
                g.add((author_uri, RDF.type, newClass))
                #g.add((author_uri, schema.name, Literal(author_name, lang='en')))
                g.add((author_uri, RDFS.label, Literal(author_name, lang='en')))

In [11]:
# Convert the 'trustfund_key' column to string
df['trustfund_key'] = df['trustfund_key'].astype(str)
df['trustfund'] = df['trustfund'].astype(str)
# Create a dictionary that maps trustfund keys to trustfund names
trustfund_dict = {}
for i, row in df.iterrows():
    if pd.isna(row['trustfund']) or pd.isna(row['trustfund_key']):
        continue
    keys = row['trustfund'].split(',')
    values = row['trustfund_key'].split(',')
    for key, value in zip(keys, values):
        trustfund_dict[key.strip()] = value.strip()

In [12]:
# Convert the 'trustfund_key' column to string
df['projectid'] = df['projectid'].astype(str)
df['projn'] = df['projn'].astype(str)
# Create a dictionary that maps trustfund keys to trustfund names
project_dict = {}
for i, row in df.iterrows():
    if pd.isna(row['projectid']) or pd.isna(row['projn']):
        continue
    keys = row['projectid'].split(',')
    values = row['projn'].split(',')
    for key, value in zip(keys, values):
        project_dict[key.strip()] = value.strip()

In [13]:
create_subclass_country('count')
create_subclass_world_bank_document('docty')
create_subclass_trustfund('trustfund', trustfund_dict)
create_subclass_project('projn')
create_subclass_authors('authors')

In [14]:
#Save as a ttl file to view in protege
#Also save as ntriples which works better for the LlamaIndex RDFReader
g.serialize('SDKG.ttl',format='turtle',prefixes = prefixes, encoding='utf-8')
g.serialize("SDKG_nt",format="nt",prefixes = prefixes, encoding='utf-8')

<Graph identifier=N0f56494983124413a9f5861d1ea5bf2c (<class 'rdflib.graph.Graph'>)>

## 3. Build a knowledge graph (KG) by populating ontology with instances of World Bank documents (each row in the metadata)


In [15]:
#Create abstract property
df['abstracts'] = df['abstracts'].astype(str).str.replace('\n', '').replace('\\n','')
abstractIs_uri = URIRef(schema + "abstractIs")
g.add((abstractIs_uri, RDF.type, RDF.Property))
g.add((abstractIs_uri, RDFS.label, Literal("Short summary of the document.")))
#Create abstract class
abstract_class = URIRef(schema + "abstract")
g.add((abstract_class, RDFS.label, Literal("Short summary of a document.")))
#Create author properties
authoredBy_uri = URIRef(schema + "authoredBy")
authored_uri = URIRef(schema + "authored")
g.add((authoredBy_uri, RDF.type, RDF.Property))
g.add((authored_uri, RDF.type, RDF.Property))
g.add((authoredBy_uri, RDFS.label, Literal("This document was authored by this author.")))
g.add((authored_uri, RDFS.label, Literal("This author wrote this document.")))
#Define 'part of' property
isPartOf_uri = URIRef(schema + "isPartOf")
g.add((isPartOf_uri, RDF.type, RDF.Property))
g.add((isPartOf_uri, RDFS.label, Literal("This entity is a part of another entity")))
#Define 'countryOfOrigin' property
countryOfOrigin_uri = URIRef(schema + "countryOfOrigin")
g.add((countryOfOrigin_uri, RDF.type, RDF.Property))
g.add((countryOfOrigin_uri, RDFS.label, Literal("Country that this document is about.")))
# Create instances for each document and add author property
for index, row in tqdm(df.iterrows()):
    if not pd.isnull(row['id']) and not pd.isnull(row['docty']) and not pd.isnull(row['authors']):
        try:
            # Create the report instance
            instance = URIRef(schema + "doc/" + str(row['display_title']).replace(" ","_").replace("-","_"))
            g.add((instance, RDFS.label, Literal(str(row['display_title']), lang='en')))            
            #Connect instances with types of documents
            doctype = URIRef(row['docty'])
            g.add((instance, RDF.type, doctype))    
            #Connect instances with country of origin
            if row['count'] != "nan":
                country = URIRef(schema + str(row['count']).replace(" ","_").replace("-","_"))
                g.add((instance, countryOfOrigin_uri, country))
            #Connect instances with projects
            if row['projn'] != "nan":
                project = URIRef(schema + "project/" + str(row['projn']).replace(" ","_").replace("-","_"))
                g.add((instance, isPartOf_uri, project))
            #Connect instances with trustfund_keys
            if row['trustfund'] != "nan":
                tf_values = row['trustfund'].split(",")
                for tf in tf_values:
                    trustfund_uri = URIRef(schema + "trustfund/" + str(tf).replace(" ","_").replace("-","_"))
                    g.add((trustfund_uri, RDFS.label, Literal(f"Trustfund: {tf}")))
                    g.add((instance, isPartOf_uri, trustfund_uri))
                    g.add((trustfund_uri, countryOfOrigin_uri, country))                
            #Connect instances with authors
            author_dict = safe_literal_eval(str(row['authors']))
            for author_dict_entries in author_dict.values():
                author_name = author_dict_entries.get('author')
                if not isinstance(author_name, str):
                    continue
                author_uri = URIRef(schema + "author/" + author_name.replace(" ", "_"))
                g.add((instance, authoredBy_uri, author_uri))
                g.add((author_uri, authored_uri, instance))
            #Add abstract
            if row['abstracts'] != "nan":
                abstract_uri = URIRef(schema + "abstract/" + str(row['display_title']).replace(" ","_").replace("-","_"))
                g.add((instance, abstractIs_uri, abstract_uri))
                g.add((abstract_uri, RDFS.label, Literal(str(row['abstracts']))))
                g.add((abstract_uri, RDF.type, abstract_class))
                g.add((abstract_uri, isPartOf_uri, abstract_uri))
        except:
            pass

101it [00:00, 5726.98it/s]


In [16]:
g.serialize('SDKG.ttl',format='turtle',prefixes = prefixes, encoding='utf-8')
g.serialize("SDKG_nt",format="nt",prefixes = prefixes, encoding='utf-8')

<Graph identifier=N0f56494983124413a9f5861d1ea5bf2c (<class 'rdflib.graph.Graph'>)>

## 4. Bring in some additional data from Wikidata


In [17]:
def get_property_label(property_code):
    # Set up the SPARQL endpoint
    sparql = make_sparql()
    # Construct the SPARQL query
    query = f"""
    SELECT ?label WHERE {{
      wd:{property_code} rdfs:label ?label.
      FILTER (lang(?label) = 'en')
    }}
    """
    # Set the query and response format
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    # Execute the query and retrieve the results
    results = run_sparql(sparql)
    # Extract and return the property label
    if 'results' in results and 'bindings' in results['results'] and results['results']['bindings']:
        label = results['results']['bindings'][0]['label']['value']
        return label
    return None

In [18]:
import numpy as np

sparql = make_sparql()
# Create a cache to store property code-label mappings
property_cache = {}
entity_cache = {}
# Prepare a list to collect triples for bulk graph update
triples = []
# Iterate over the URIs and add the properties to the RDF graph
for uri in tqdm(df['country_URI']):        
    if isinstance(uri, str) and uri.startswith('http://www.wikidata.org/entity/Q'):
        class_uri = URIRef(uri)
        country_column = df.loc[df['country_URI'] == uri, 'count'].iloc[0]
        country_column = URIRef(schema + sanitize_uri_component(country_column))
        # Construct the SPARQL query
        qid = uri.split('/')[-1]
        query = f"""
        SELECT ?prop ?value WHERE {{
          wd:{qid} ?prop ?value .
          OPTIONAL {{ ?prop rdfs:label ?label . FILTER(lang(?label) = 'en') }}
        }}
        """
        # Set the query and response format
        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)
        # Execute the query and retrieve the results
        results = run_sparql(sparql)
        # Iterate over the results and add them to the RDF graph
        for result in results["results"]["bindings"]:
            prop = result["prop"]["value"]
            value = Literal(result["value"]["value"])
            triple = (country_column, None, None)  # Placeholder for triple            
            if prop.startswith('http://www.wikidata.org/prop'):
                property_code = prop.split('/')[-1]
                # Check if the property code is already in the cache
                if property_code in property_cache:
                    property_label = property_cache[property_code]
                else:
                    # If not in cache, query and retrieve the property label
                    property_label = get_property_label(property_code)
                    # Store the property code-label mapping in the cache
                    property_cache[property_code] = property_label                
                property_label_URI = URIRef(schema + sanitize_uri_component(property_label))
                triple = (country_column, property_label_URI, value)            
            if value.startswith('http://www.wikidata.org/entity/Q'):
                entity_code = value.split('/')[-1]
                # Check if the entity code is already in the cache
                if entity_code in entity_cache:
                    entity_label = entity_cache[entity_code]
                else:
                    # If not in cache, query and retrieve the entity label
                    entity_label = get_entity_label(entity_code)
                    # Store the entity code-label mapping in the cache
                    entity_cache[entity_code] = entity_label
                entity_label_URI = URIRef(schema + sanitize_uri_component(entity_label))
                triple = (country_column, property_label_URI, entity_label_URI)            
            triples.append(triple)        
    elif isinstance(uri, float) and np.isnan(uri):
        continue
    else:
        continue

# Add all collected triples to the RDF graph in bulk
for subject, predicate, object_ in triples:
    if predicate is not None:
        g.add((subject, predicate, object_))

  2%|▏         | 2/101 [06:16<5:10:18, 188.07s/it]

HTTP 429, waiting 120s (attempt 1/6)...
HTTP 429, waiting 120s (attempt 1/6)...
HTTP 429, waiting 120s (attempt 1/6)...
HTTP 429, waiting 120s (attempt 1/6)...
HTTP 429, waiting 120s (attempt 1/6)...


  8%|▊         | 8/101 [36:47<4:27:00, 172.26s/it] 

HTTP 429, waiting 120s (attempt 1/6)...
HTTP 429, waiting 120s (attempt 1/6)...


100%|██████████| 101/101 [1:09:06<00:00, 41.05s/it]


In [ ]:
g.serialize('SDKG.ttl',format='turtle',prefixes = prefixes, encoding='utf-8')
#g.serialize("SDKG_nt",format="nt",prefixes = prefixes, encoding='utf-8')

<Graph identifier=N213871fefdbd4948b878115aba5fa456 (<class 'rdflib.graph.Graph'>)>

## 5. Query KG using SPARQL


In [1]:
import rdflib
g = rdflib.Graph()
g.parse("SDKG.ttl", format="turtle")

<Graph identifier=N8e83fbfa875a48f895c030023c9c8d6c (<class 'rdflib.graph.Graph'>)>

In [2]:
# Step 1: Find the URI of Brazil in your ontology
brazil_uri = "<http://schema.org/Brazil>"  # Replace with the actual URI
# Step 2: Find the most relevant documents related to Brazil
documents_query = f"""
PREFIX schema: <http://schema.org/>
SELECT ?document
WHERE {{
  ?document a/rdfs:subClassOf* schema:world_bank_document ;
      schema:countryOfOrigin {brazil_uri} .

}}
"""
qres = g.query(documents_query)
for row in qres:
    print(f"Document ID: {row.document}")

Document ID: http://schema.org/doc/Brazil___LATIN_AMERICA_AND
____________CARIBBEAN___P167455___Ceara_Rural_Sustainable_Development
____________and_Competitiveness_Phase_II___Audited_Financial_Statement
Document ID: http://schema.org/doc/Environmental_and_Social
____________Commitment_Plan_(ESCP)_Mato_Grosso_Sustainable_Development
____________of_Family_Farming_(P175723)
Document ID: http://schema.org/doc/Environmental_and_Social
____________Commitment_Plan_(ESCP)___Rio_Grande_do_Norte:_Sustainable
____________Development_and_Governance_of_Rio_Grande_do_Norte___Phase_2
______________P502493
Document ID: http://schema.org/doc/Revised__Resettlement_Framework
____________Rio_Grande_do_Norte:_Sustainable_Development_and_Governance
____________of_Rio_Grande_do_Norte___Phase_2__(P502493)
Document ID: http://schema.org/doc/Revised___Environmental_and
____________Social_Management_Framework_(ESMF)_Rio_Grande_do_Norte:
____________Sustainable_Development_and_Governance_of_Rio_Grande_do
________

In [3]:
from rdflib import Graph, RDF, RDFS, URIRef
# Step 1: Find the URI of the basic_form_of_government you are interested in
government_form_uri = "<http://schema.org/federal_republic>"  # Replace with the actual URI
# Step 2: Query for authors who have written the most documents associated with countries having the basic_form_of_government as "federal_republic"
authors_query = f"""
PREFIX schema: <http://schema.org/>
PREFIX prop: <http://schema.org/property>
SELECT ?author (COUNT(?document) AS ?numDocuments)
WHERE {{
  ?document a/rdfs:subClassOf* schema:world_bank_document ;
            schema:countryOfOrigin [
                    schema:basic_form_of_government {government_form_uri}
            ] ;
            schema:authoredBy ?author .
}}
GROUP BY ?author
ORDER BY DESC(?numDocuments)
"""
# Execute the query
results = g.query(authors_query)
# Now you can process the results and present them as needed (e.g., using pandas DataFrames)
# For simplicity, here, I'm just printing the author names and the number of documents they wrote
for row in results:
    print(f"Author: {row.author}, Number of Documents: {row.numDocuments}")

Author: http://schema.org/author/Chichgar,Sharlene_Jehanbux, Number of Documents: 3
Author: http://schema.org/author/Borrowing_Agency, Number of Documents: 2
Author: http://schema.org/author/Ru,Jiang, Number of Documents: 2
Author: http://schema.org/author/Ramirez,Maria_Catalina, Number of Documents: 2
Author: http://schema.org/author/D_Agostino_Fontaine,Santiago_Manuel, Number of Documents: 1
Author: http://schema.org/author/Noronha_Farinelli,Barbara_Cristina, Number of Documents: 1
Author: http://schema.org/author/Colbano,Fabiano_Silvio, Number of Documents: 1
Author: http://schema.org/author/Daniel_Larrache, Number of Documents: 1
Author: http://schema.org/author/Lucas_Figueras, Number of Documents: 1


## 6. Compare the different ways of interacting with data: using SPARQL queries on the RDF, putting raw metadata into LlamaIndex, and putting the RDF data into LlamaIndex


### Putting raw data into LlamaIndex

In [5]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import os

# OLLAMA_BASE_URL in .env is the OpenAI-compatible endpoint (ends in /v1) used by my_graphrag.py's
# AsyncOpenAI client. llama-index's Ollama integrations talk to Ollama's native API instead, so strip it.
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").removesuffix("/v1")

Settings.llm = Ollama(model="gpt-oss:20b", base_url=OLLAMA_BASE_URL, request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text", base_url=OLLAMA_BASE_URL)

#Make sure your raw.csv file is in the data folder
documents = SimpleDirectoryReader('data').load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [6]:
response = query_engine.query("Show me all of the World Bank documents in the context information about Brazil")
print(response)

**World Bank documents that refer to Brazil**

| Document | Link |
|----------|------|
| Text of the Revised Environmental and Social Management Framework (ESMF) – Rio Grande do Norte (Phase 2) | <http://documents.worldbank.org/curated/en/099071924123593973/text/P5024931750b5d0d18c3f13400756ccc9d.txt> |
| Project‑wide document path for the same ESMF (Phase 2) | <https://projects.worldbank.org/projects-documents/2024/07/34366675/revised-resettlement-framework-rio-grande-norte-sustainable-development-governance-rio-grande-norte-phase-2-p502493> |
| Web page for the ESMF (Phase 2) | <http://documents.worldbank.org/curated/en/099071924123593973/P5024931750b5d0d18c3f13400756ccc9d> |
| PDF of the revised Environmental and Social Management Framework (Phase 2) | <http://documents.worldbank.org/curated/en/099071924123085917/pdf/P50249313cc4e1091b98e18ead67a6c02c.pdf> |
| Text version of the same PDF | <http://documents.worldbank.org/curated/en/099071924123085917/text/P50249313cc4e1091b98e18ead

In [7]:
response = query_engine.query("Based on the context information, what documents has Corsi,Anna written?")
print(response)

There are no documents attributed to Corsi, Anna in the provided data.


In [8]:
response = query_engine.query("Tell me more about Anna Corsi")
print(response)

I’m sorry, but I don’t have any information on Anna Corsi.


In [9]:
response = query_engine.query("Tell me more about World Bank's land management infrastructure project in Turkey")
print(response)

The World Bank’s **Land Management Infrastructure for Green and Sustainable Development** project in Turkey (Project ID P179217) is a procurement‑led initiative designed to strengthen the country’s capacity for environmentally‑sound land use.  

**Key focus areas**  
- **Green infrastructure** that supports climate‑resilient land management.  
- **Geospatial‑based planning** to improve disaster preparedness and resource mapping.  
- **Public‑sector capacity building** in land administration, municipal governance, and ICT solutions.  
- **Cross‑cutting themes** such as gender, rural development, finance, and private‑sector engagement.  

**Financing and structure**  
- The project is financed under the World Bank’s Investment Project Financing framework, enabling the procurement of goods, services, and technical assistance needed to deliver the infrastructure components.  
- It is part of the **Europe and Central Asia** regional portfolio (Turkiye – EUROPE AND CENTRAL ASIA).  

**Docume

### Putting RDF data into LlamaIndex

In [ ]:
#llama-index-core comes pre-bundled with NLTK and tiktoken files to avoid downloads and network calls at runtime.
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.schema import Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from rdflib import Graph
from collections import defaultdict
import os

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434").removesuffix("/v1")
Settings.llm = Ollama(model="gpt-oss:20b", base_url=OLLAMA_BASE_URL, request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text", base_url=OLLAMA_BASE_URL)

# llama_index's download_loader()/RDFReader (llama-hub) and LLMPredictor/ServiceContext no longer
# exist in current llama-index-core, and no RDF reader package replaces them, so parse the graph
# with rdflib directly. SDKG_nt's strict N-Triples round-trip chokes on some URIs built earlier in
# the notebook, so load SDKG.ttl (Turtle) instead, which already parses cleanly in step 5.
rdf_graph = Graph()
rdf_graph.parse("SDKG.ttl", format="turtle")
triples_by_subject = defaultdict(list)
for s, p, o in rdf_graph:
    triples_by_subject[str(s)].append(f"{p} {o}")
document = [
    Document(text="\n".join(triples), doc_id=subject)
    for subject, triples in triples_by_subject.items()
]
index = VectorStoreIndex.from_documents(document)
query_engine = index.as_query_engine()

In [12]:
response = query_engine.query("Show me all of the World Bank documents in the context information about Brazil")
print(response)

**World Bank documents related to Brazil in the provided context**

- *Ceará Rural Sustainable Development and Competitiveness Phase II – Audited Financial Statement* (Auditing Document for the Brazilian project P167455)


In [13]:
response = query_engine.query("Based on the context information, what documents has Corsi,Anna written?")
print(response)

Corsi, Anna has authored two documents titled **“Disclosable Version of the ISR__ Land Management Infrastructure for Green and Sustainable Development”**, specifically the sections identified as:

1. **P179217 Sequence No.: 3**  
2. **P179217 Sequence No.: 4**


In [14]:
response = query_engine.query("Tell me more about Anna Corsi")
print(response)

Anna Corsi is credited as an author of the document titled **“Disclosable Version of the ISR__Land management infrastructure for green and sustainable development.”** The record shows two references to this work, indicating she contributed to its creation and publication.


In [15]:
response = query_engine.query("Tell me more about World Bank's land management infrastructure project in Turkey")
print(response)

The World Bank’s land management infrastructure initiative in Turkey, identified by the project code P179217, is designed to support green and sustainable development across the country. The program falls under the broader “TR Land Management Infrastructure for Green and Sustainable Development” umbrella and is focused on improving the planning, execution, and monitoring of land‑use projects to promote environmental resilience and sustainable growth.

Key components of the initiative include:

* **Implementation Status and Results Report (ISR)** – A disclosable version of the ISR tracks progress, outcomes, and lessons learned, providing transparency on how project funds are being used and the impact on Turkey’s land‑management landscape.
* **Procurement Plan** – Developed to guide the acquisition of goods, services, and works essential for infrastructure development, ensuring timely delivery and adherence to World Bank procurement standards.
* **Authorship** – The ISR and procurement p

### Try using both RDF and raw data to see if it helps

In [16]:

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
#Make sure to update the files in the data folder
documents = SimpleDirectoryReader('data').load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

In [17]:
response = query_engine.query("Show me all of the World Bank documents in the context information about Brazil")
print(response)

**World Bank documents related to Brazil**

| Document ID | Project | Document type | Title | Language | PDF link | Text link |
|-------------|---------|---------------|-------|----------|----------|-----------|
| 34366674 | P502493 | Environmental and Social Management Plan | *Revised Environmental and Social Management Framework (ESMF) Rio Grande do Norte: Sustainable Development and Governance of Rio Grande do Norte – Phase 2* | Portuguese | <http://documents.worldbank.org/curated/en/099071924123085917/pdf/P50249313cc4e1091b98e18ead67a6c02c.pdf> | <http://documents.worldbank.org/curated/en/099071924123085917/text/P50249313cc4e1091b98e18ead67a6c02c.txt> |

These are the only World Bank documents in the provided data that pertain to Brazil.


In [18]:
response = query_engine.query("Based on the context information, what documents has Corsi,Anna written?")
print(response)

None.


In [19]:
response = query_engine.query("Tell me more about Corsi,Anna")
print(response)

I’m sorry, but I don’t have any information about Corsi, Anna in the data you provided.


In [20]:
response = query_engine.query("Tell me more about World Bank's land management infrastructure project in Turkey")
print(response)

The World Bank’s **Land Management Infrastructure for Green and Sustainable Development (Project P179217)** is a Turkey‑focused initiative under the Europe and Central Asia region. Its goal is to modernise land‑use planning and governance by integrating property and geospatial data into disaster‑resilience, climate‑adaptation and smart‑city frameworks.  

**Key Elements**

| Feature | Detail |
|---------|--------|
| **Title** | Land Management Infrastructure for Green and Sustainable Development (Turkey) |
| **Project ID** | P179217 |
| **Funding** | IBRD/IDA (public) |
| **Themes** | Public administration, mitigation, municipal institution building, human development, ICT, rural and social development, environment and natural resources, finance, private‑sector development, land administration, climate change, urban & rural development, adaptation, geospatial services, civil‑service reform |
| **Sub‑projects** | • Leveraging property and geospatial data for disaster and climate resilie